# ICS 235 Assignment 3
## Comparing Classifiers: Naïve Bayes, SVM, and Decision Trees
# <span style="color:red">Due: 11:59 PM on Monday, November 2</span>

## Instructions

1. Rename your notebook file using the format: **`<LastName><FirstName>_3.ipynb`**.
    - For example, if your first name is John and your last name is Smith, the filename should be `SmithJohn_3.ipynb` (the number at the end is the assignment number).
2. Only use the `.ipynb` file extension. Other formats (`.rtf`, `.zip`, `.docx`, `.pdf`) are not accepted.
3. The data file will be available to the instructor and the TA, so there is no need to upload it. Make sure you use the same filename given in the assignment.
4. Save the data file in your **working directory**.
5. **Do not modify or delete the provided code and the code cell.**
   - Do not add your own code to the existing code cell containing the provided code.
   - Keep the provided code unchanged so that it can be used as a reference and checked as intended.
7. Clean your code before submission.
    - If needed, provide clear documentation describing the purpose and use of every class or function you write.
    - Your submission should **show only the required outputs**.
8. **<font color='red'>IMPORTANT:</font>** Before submitting your homework:
   - Save the notebook.
   - Restart the kernel to clear its memory and **run the whole notebook top to bottom.** (`Kernel → Restart & Run All Cells`).
   - Confirm every cell completes without error and all outputs are visible.
9. Write your full name in the cell below.


### AI tools

You may use AI coding assistants as a learning aid, but **you must understand and be able to explain every line you submit**. You may be asked to do so. Submitting code you cannot explain is an academic integrity violation.
***

## Your Name: Denny Huang
***

# Getting Started

<font color='blue'>**Do not modify any of the provided code in this section. Run the code exactly as provided.**</font> 
- If you are using Google Colab, you may add a **separate code cell** for loading the data. Do not add the data-loading code to the provied code cell.
  
## About the data

This assignment uses `pa3_oahu_commute.csv`, a survey of how people on Oʻahu get to work.

> **This dataset is synthetic.** It was generated for instructional use in this course. The relationships are plausible, but no row corresponds to a real household, and nothing here should be used to draw conclusions about actual commuting on Oʻahu.

The table has **900 rows** and **8 columns**:

| Column | Description |
|---|---|
| `dist_to_work_km` | Distance from home to workplace, in kilometres |
| `median_income_k` | Median household income for the area, in thousands of dollars |
| `household_size` | Number of people in the household |
| `has_car` | 1 if the household owns a car, 0 otherwise |
| `dist_to_busstop_m` | Walking distance to the nearest bus stop, in metres |
| `age` | Age of the respondent |
| `downtown_worker` | 1 if the workplace is in downtown Honolulu, 0 otherwise |
| `commute_mode` | **The target.** One of `drive`, `bus`, `bike`, `walk` |

## The task

Predict a respondent's `commute_mode` from the other seven columns.

1. **There are four classes.** Binary metrics like precision and recall have to be computed *per class* and then combined, and how you combine them turns out to change the answer.
   
2. **You will compare three different classifiers** — Naïve Bayes, a support vector machine, and a decision tree — rather than tuning one. Each makes a different assumption about the data, and you will find out what those assumptions cost.

Download `pa3_oahu_commute.csv` from Lamaku and save the file in your **working directory**.

## Importing packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility: use this constant everywhere a random_state is required.
RANDOM_STATE = 42

## Loading the dataset

In [2]:
commute = pd.read_csv("pa3_oahu_commute.csv")
commute.head()

,dist_to_work_km,median_income_k,household_size,has_car,dist_to_busstop_m,age,downtown_worker,commute_mode
0,12.1,62.1,3,1,458.0,32,0,drive
1,0.6,68.1,1,1,167.0,42,0,walk
2,3.7,96.6,3,1,39.0,44,0,drive
3,19.9,85.3,2,0,423.0,45,0,bus
4,13.6,25.0,2,1,20.0,26,0,drive


The next cell defines the column groups and the *class ordering* you will use throughout. Run it
now.

In [3]:
features = ["dist_to_work_km", "median_income_k", "household_size", "has_car",
            "dist_to_busstop_m", "age", "downtown_worker"]
TARGET = "commute_mode"

# The four modes, ordered most to least common. Use this ordering whenever you
# tabulate or plot by class -- it makes confusion matrices easier to read than
# the default alphabetical order.
modes = ["drive", "bus", "bike", "walk"]

***
# Exercises

This assignment has **three parts**, worth 100 points total.

Write your answer wherever you see a *`# YOUR CODE`* comment in a code cell or a
**`YOUR ANSWER:`** prompt in a markdown cell. Some cells already contain code to get you
started — **do not modify the provided code**. The point value of each question is given in
parentheses after the question text.

| Part | Topic | Points |
|---|---|---|
| 1 | The data and the multiclass baseline | 29 |
| 2 | Three classifiers, and what each one assumes | 39 |
| 3 | Choosing between them | 32 |

***
## Part 1: The Data and the Multiclass Baseline (29 points)

### Question 1. Dataset structure (1 + 1 + 1 + 2 = 5 points)

1. Print the number of rows and columns in a readable sentence.
2. Display the column names with their data types and non-null counts.
3. Display the **first 10 rows** of the dataset.
4. In the answer cell: The features `has_car` and `downtown_worker` are stored as integers. Should they be treated as numerical or categorical features? Does the distinction matter in this case? Briefly explain.

In [15]:
# YOUR CODE
rows, cols = commute.shape[0], commute.shape[1]
print(f"Row count: {rows}\nColumn count: {cols}")

display(commute.info())
display(commute.head(10))

# display(commute['has_car'].value_counts())
# display(commute['downtown_worker'].value_counts())

Row count: 900
Column count: 8
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   dist_to_work_km    900 non-null    float64
 1   median_income_k    884 non-null    float64
 2   household_size     900 non-null    int64  
 3   has_car            900 non-null    int64  
 4   dist_to_busstop_m  886 non-null    float64
 5   age                900 non-null    int64  
 6   downtown_worker    900 non-null    int64  
 7   commute_mode       900 non-null    str    
dtypes: float64(3), int64(4), str(1)
memory usage: 56.4 KB


None

,dist_to_work_km,median_income_k,household_size,has_car,dist_to_busstop_m,age,downtown_worker,commute_mode
0,12.1,62.1,3,1,458.0,32,0,drive
1,0.6,68.1,1,1,167.0,42,0,walk
2,3.7,96.6,3,1,39.0,44,0,drive
3,19.9,85.3,2,0,423.0,45,0,bus
4,13.6,25.0,2,1,20.0,26,0,drive
5,20.8,29.7,4,1,124.0,43,0,drive
6,20.8,73.6,3,1,167.0,24,1,drive
7,2.1,79.1,1,0,47.0,38,0,bus
8,32.5,NaN,5,1,20.0,44,1,drive
9,18.3,86.1,1,0,483.0,18,1,bus


>**YOUR ANSWER (Q1.4):** They should be treated as categorical features because their values (0 and 1) represent categories, whether a person has a car (1) or not (0), and whether a person works in downtown (1) or not (0). The stored integers are not technically numerical quantities. In this case, the distinction doesn't matter too much because each feature already has only 2 values. Using the column as a number is the same as using an indicator for "Yes" or "True" and "No" or "False", so the model reflects the same two groups either way.
>
>

### Question 2. Four classes and the baseline (1 + 3 + 2 = 6 points)

1. Print the count and proportion of observations in each `commute_mode` class.
2. Compute, **in code**, the accuracy of a baseline classifier that always predicts the most common communte mode. Report the baseline accuracy and the commute mode it predicts.
3. In the answer cell, address the following questions:
   - With four classes, you might expect a baseline accuracy of approximately 1/4 = 25%. Explain why the baseline accuracy is much higher than 25% in this dataset.
   - Which two classes do you expect a model to have the most difficulty classifying correctly, and why?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q2.3):**
>
>

### Question 3. Missing values (1 + 1 + 1 = 3 points)

1. Print the count and percentage of missing values for every column that has any.
2. Report how many rows have at least one missing value.
3. In the answer cell, explain how you will handle missing values in the features before training your model. Describe your approach and justify your choices.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q3.3):**
>
>

### Question 4. Which features separate the four modes? (1 + 3 + 2 + 3 = 9 points)

Use the same technique as previous assignments: standardize first, so the features become comparable, then compare the class means.

1. Standardize all seven features by subtracting each feature’s mean and dividing by its standard deviation.
2. Compute and print the **means of each standardized feature for each `commute_mode` class**. Order the rows according to `modes` defined in the `Getting Started` section.
   - You may use `DataFrame.groupby()` or calculate the group means manually.
3. For each feature, calculate the **spread of its standardized class means**, defined as the largest class mean minus the smallest class mean. Print the feature names and their spread values, sorted by in descending order of spread.
4. In the answer cell: identify the **three most useful features** based on these results. For each feature, explain which commute modes it helps separate and which classes it may not distinguish well. A feature may be informative for separating some classes while less useful for distinguishing others.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q4.4):**
>
>

### Question 5. Visualize the structure (2 + 1 + 3 = 6 points)

1. Create a **scatter plot** with `dist_to_work_km` on the x-axis and `dist_to_busstop_m` on the y-axis. **Color the points by `commute_mode`** by looping over `modes` and passing `label=mode` to the plotting call.
2. Label both axes, add a descriptive title and a legend, and set the point transparency (alpha) to 0.7.
3. In the answer cell, interpret the plot:
   - What patterns or clusters do you observe among the four commute modes?
   - Could a single vertical or horizontal line separate any of the classes? Explain why or why not.
   - Which classes overlap the most, and how does this compare with your findings in Question 4?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q5.3):**
>
> 

***
## Part 2: Three Classifiers and What Each One Assumes (39 points)

Each of the three models you are about to fit makes a different assumption about how the data
is shaped. The point of this part is to find out what those assumptions buy and what they cost.

Run the provided setup cell first. It builds **two** preprocessors — one with scaling and one
without — because you will need both in Question 4.

In [ ]:
# ============================ PROVIDED SETUP — do not modify ============================
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier

X = commute[features]
y = commute[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

# Impute + scale (what we would normally use)
preprocess = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())])

# Impute only -- NO scaling. Needed for Question 4.
preprocess_unscaled = SimpleImputer(strategy="median")

# Baseline accuracy on the held-out test set
baseline_acc = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_test, y_test)
print(f"Training set: {len(X_train)} rows    Test set: {len(X_test)} rows")
print(f"Baseline accuracy on the test set: {baseline_acc:.4f}")
print(f"Test-set class counts:\n{y_test.value_counts().reindex(modes).to_string()}")
# =======================================================================================

### Question 1. Naïve Bayes and its independent assumption (2 + 5 + 3 = 10 points)

1. Fit a **Gaussian Naïve Bayes** classifier in a pipeline using `preprocess`. Report its test accuracy alongside the baseline classifier's test accuracy.
   
2. Naïve Bayes assumes that the features are **conditionally independent given the class**. In other words, once the commute mode is known, the model assumes that knowing one feature provides no additional information about another feature. For example, among people who drive, it assumes that income and distance to the nearest bus stop are independent.<br><br>
   To investigate this assumption, calculate the feature correlation matrix **separately for each commute mode** and report the largest absolute off-diagonal correlation for each class. Follow these steps:
   - Loop over `modes` and select the rows belonging to each class.
   - Compute the correlation matrix using `.corr()`.
   - Take the absolute values using `.abs()` and ignore the diagonal, which contains self-correlations.
   - Report the **largest absolute off-diagonal correlation** and the **corresponding pair of features for each class**.<br><br>

3. In the answer cell, address the following questions:
   - Is the conditional independence assumption reasonable for this dataset, based on the correlations you found?
   - Which commute mode shows the strongest evidence of dependence between two features? Identify the feature pair and explain why their relationship makes sense in real-world terms.
   - Naïve Bayes' independence assumption is rarely perfectly satisfied in real datasets. Why can the model still be useful despite this limitation?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q1.3):**
>
>

### Question 2. Support vector machines: choosing a kernel and tuning $C$ (5 + 2 + 3 = 10 points)

1. Using **5-fold cross-validation on the training set**, compare `SVC` models with `kernel="linear"` and `kernel="rbf"`, each tested with `C` values of `0.1`, `1`, `10` and `100`.
   - Use a pipeline that includes `preprocess`.
   - Set `random_state=RANDOM_STATE` for each SVC model.
   - Report the **mean cross-validation accuracy** for all eight combinations.
2. Select the combination with the highest mean cross-validation accuracy, fit it on the full training set, and report its **accuracy on the held-out test set**.
3. In the answer cell, address the following questions:
   - Which kernel performs better based on cross-validation accuracy? What does this suggest about whether the classes can be separated effectively by a linear decision boundary or require a more flexible, nonlinear boundary?
   - The parameter `C` controls how strongly the SVM penalizes training errors. How do very small and very large values of `C` affect the decision boundary, and how might they lead to underfitting or overfitting?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q2.3):**
>
>

### Question 3. Decision trees: depth and what the tree tells you (3 + 1 + 2 + 3 = 9 points)

1. Using **5-fold cross-validation on the training set**, compare `DecisionTreeClassifier` models with `max_depth` values of `[2, 3, 4, 5, 6, 8, 10, None]`.
   - Use a pipeline that includes `preprocess`.
   - Set `random_state=RANDOM_STATE` for each decision tree.
   - Report the **mean cross-validation accuracy** for all eight depth settings.
2. Select the depth with the highest mean cross-validation accuracy, fit the selected model on the full training set, and report its **accuracy on the held-out test set**.
3. Print the selected tree’s **feature importances**, sorted from highest to lowest, with the corresponding feature names.
4. In the answer cell, address the following questions:
   - Which tree depth performs best based on cross-validation accuracy? What happens to performance as the tree becomes deeper?
   - How do the tree’s feature importances compare with your feature ranking from Question 4 in Part 1? Do the two methods identify similar important features? Explain any differences.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q3.4):**
>
>

### Question 4. Which of these models care about feature scaling? (4 + 2 + 4 = 10 points)

In Assignment 1, removing the scaler substantially reduced k-NN performance. In Assignment 2, it caused gradient descent to diverge. In this question, you will investigate how feature scaling affects the three classifiers from this assignment.

1. Fit each of the three classifiers **twice**, using the **best hyperparameter settings** identified in the previous questions:
   - **With scaling:** use `preprocess` (imputation + standardization)
   - **Without scaling:** use `preprocess_unscaled` (imputation only).<br><br>
   
   Set `random_state=RANDOM_STATE` for both the SVM and the decision tree.
   
   Report the **test accuracy** of all six fitted models in a clearly labeled table.<br>
3. Identify the model whose test accuracy changes the most when scaling is removed. For that model, print the unique classes predicted by the unscaled version.
   - Hint: use `np.unique(predictions)`.<br><br>
     
4. In the answer cell, explain the results for **each of the three classifiers**:
   - Which classifier(s) are most affected by removing feature scaling, and why?
   - Which classifier(s) are largely unaffected by removing feature scaling, and why?
   - Explain the behavior of each algorithm based on how it uses feature values, distances, or decision boundaries. Do not simply state that an algorithm "needs scaling"; explain the underlying reason.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q4.3):**
>
>

***
## Part 3: Choosing Between Them (32 points)

You have now three fitted classifiers. In this part, you will compare their performance and consider how the choice of evaluation metric affects which model appears best.

### Question 1. Accuracy and macro-F1: Do they agree? (4 + 2 + 4 = 10 points)

With four classes, precision, recall, and F1 can be calculated separately for each class and then averaged across classes. The averaging method matters:
- **Macro-F1** calculates the F1 score for each class and takes their unweighted average, giving every class equal importance regardless of its size.
- **Weighted-F1** averages the per-class F1 scores, weighting each by the number of true observations in that class. As a result, larger classes have more influence on the score.

1. For all three classifiers, report **test accuracy**, **macro-F1** and **weighted-F1** in a clearly labeled table.
   - Use the best models fitted in **Questions 1, 2, and 3 of Part 2**. **Do not retrain the models here.**
   - Hint: use `f1_score(y_true, y_pred, average="macro")` and `f1_score(y_true, y_pred, average="weighted")`.
2. Rank the three models from best to worst by test accuracy, then rank them by test macro-F1 score.
3. In the answer cell, address the following questions:
   - Do the two rankings agree? How do their scores compare?
   - What does macro-F1 reveal about model performance that accuracy alone may miss?
   - Which metric — accuracy, macro-F1, or weighted-F1 — is most appropriate for evaluating these classifiers on this dataset? Justify your choice with reference to the class distribution and the goal of the classification task.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q1.3):**
>
>

### Question 2. Interpreting the confusion matrices (2 + 2 + 4 = 8 points)

1. For each of the three fitted models used in Question 1, display the **confusion matrix for the test-set prediction** as a labeled `DataFrame`, with rows and columns in `modes` order (not alphabetical).
2. For each of the three fitted models, print the **classification report for the test-set predictions**, using `digits=3`.
3. In the answer cell, address the following questions:
   - Which pair of classes is most frequently confused by the model with the highest macro-F1, and did your analysis in Part 1 predict this difficulty?
   - Examine the `walk` row in Naïve Bayes's confusion matrix alongside its classification report. Its precision is high, but its recall is very low. What does this tell you about the model's predictions for the `walk` class?
   - All three models tend to predict the more common classes more often. Why might this happen, given the class distribution and how the models are trained?

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q2.3):**
>
>

### Question 3. Give all classes equal importance (2 + 2 = 4 points)

Both `SVC` and `DecisionTreeClassifier` support `class_weight="balanced"`, which gives greater weight to examples from less frequent classes during training. The weights are inversely proportional to class frequencies, so a `walk` observation receives roughly seven times the weight of a `drive` observation.

1. Refit the SVM and decision tree using `class_weight="balanced"`, **keeping the best hyperparameter settings** identified in Part 2. Report their **test accuracy** and **macro-F1** alongside the corresponding scores from the original models.
2. Explain how accuracy and macro-F1 changed for each model. How do these results relate to your explanation in Question 2 for why the models tend to favor common classes?

Note: `GaussianNB` does not support a `class_weight` parameter; it uses explicit class priors instead. You may note this distinction, but you do not need to implement a balanced-prior version.

In [ ]:
# YOUR CODE



>**YOUR ANSWER (Q3.2):**
>
>

### Question 4. Which model would you deploy, and what would you tell people? (10 points)

Imagine the Oʻahu Department of Transportation wants to use a model to estimate commute-mode usage in neighbourhoods where demographic data are available but survey data are not.

Write **250–400 words** in plain language for a transportation planner. Do not include code or use technical jargon without explaining it. Address all five points:

1. **Purpose and performance:** Explain what the model does and how well it performs compared with the baseline.
2. **Model choice:** Which of the three models would you deploy, and why? Consider not only predictive performance but also factors such as speed, interpretability, and the assumptions each model makes about the data.
3. **Limitations:** Identify specifically which commute modes or situations the model struggles with, and explain whether and how these limitations could be addressed.
4. **Appropriate use:** Identify one task or decision for which you would not trust the model, given the classification errors you found in Question 2. Explain why.
5. **Next steps:** Recommend one specific action you would take before or after deployment, and explain why it could improve the model or its practical use.

>**YOUR ANSWER:**
>
>